# PyDI Data Integration Workflow: Videogames

This notebook demonstrates comprehensive data integration using PyDI. We'll work with vidoegame datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 0: Schema Matching and Normalization](#part-0-schema-matching-and-normalization)
- [Part 1: Data Profiling](#part-1-data-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

### Datasets

- **DBpedia**: 65,000 records
- **Metacritic**: 20,494 records
- **Global Sales Ranking**: 7,877 records

## Part 0: Schema Matching and Normalization

In [1]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input" / "games"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "games"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Target Schema and Normalization Spec

In [3]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")
target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,releaseYear,datetime
3,developer,string
4,publisher,string
5,platform,string
6,criticScore,float
7,userScore,float
8,ESRB,string
9,globalSales,int


## Step 2: Load Source Datasets

In [4]:
dbpedia = pd.read_xml(INPUT_DIR / "schemamatching" / "original_data" / "dbpedia.xml")
dbpedia.attrs["dataset_name"] = "dbpedia"
dbpedia.head()

,gameLabel,platform,developer,genre,releaseDate,series
0,San Francisco Rush 2049,Game Boy Color,Handheld Games,Racing video game,2006-02-17,Rush (video game series)
1,RoboCop (1988 video game),Arcade video game,Ocean Software,Beat 'em up,1989-12-12,List of RoboCop video games
2,Air (video game),PlayStation Vita,Key (company),Eroge,2016-09-08,None
3,Fallout 2,Mac OS X,Black Isle Studios,Role-playing video game,1998-10-29,Fallout (series)
4,SpongeBob SquarePants: Creature from the Krust...,Wii,Blitz Games,Platform game,2006-10-18,SpongeBob SquarePants video games


In [5]:
# Drop duplicates in noisy dbpedia dataset
# drop duplicates that share name, platform, developer and releaseDate
dbpedia = dbpedia.drop_duplicates(subset=["gameLabel", "platform", "developer", "releaseDate"])
len(dbpedia)

54333

In [6]:
metacritic = pd.read_csv(INPUT_DIR / "schemamatching" / "original_data" / "metacritic.csv")
metacritic.attrs["dataset_name"] = "metacritic"
metacritic.head()

,name,release_date,developer,platform,genres,number_of_players,rating,metascore,user_score
0,Red Dead Redemption 2,"Oct 26, 2018",Rockstar Games,Xbox One,"Action Adventure,Open-World",Up to 32,M,97.0,8.3
1,Grand Theft Auto IV,"Apr 29, 2008",Rockstar North,Xbox 360,"Action Adventure,Modern,Modern,Open-World",1 Player,M,98.0,8.0
2,SoulCalibur,"Sep 8, 1999",Namco,Dreamcast,"Action,Fighting,3D",1-2,T,98.0,8.4
3,Tony Hawk's Pro Skater 2,"Sep 20, 2000",Neversoft Entertainment,PlayStation,"Sports,Alternative,Skateboarding",1-2,T,98.0,7.5
4,Super Mario Galaxy,"Nov 12, 2007",Nintendo,Wii,"Action,Platformer,Platformer,3D,3D",No Online Multiplayer,E,97.0,9.1


In [7]:
sales = pd.read_json(INPUT_DIR / "schemamatching" / "original_data" / "sales.json")
sales.attrs["dataset_name"] = "sales"
sales.head()

,Title,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76,51,8,322.0,Nintendo,E
1,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82,73,8.3,709.0,Nintendo,E
2,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80,73,8,192.0,Nintendo,E
3,New Super Mario Bros.,DS,2006,Platform,Nintendo,11.28,9.14,6.50,2.88,29.80,89,65,8.5,431.0,Nintendo,E
4,Wii Play,Wii,2006,Misc,Nintendo,13.96,9.18,2.93,2.84,28.92,58,41,6.6,129.0,Nintendo,E


In [8]:
# Create id columns based on index (starting with 1)
sales["id"] = sales.index + 1
metacritic["id"] = metacritic.index + 1
dbpedia["id"] = dbpedia.index + 1

## Step 3: LLM-Based Schema Matching

In [9]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match dbpedia dataset
dbpedia_mapping = matcher.match(dbpedia, df_target)

dbpedia_mapping

Invalid column mapping: genre -> genres


,source_dataset,source_column,target_dataset,target_column,score,notes
0,dbpedia,gameLabel,target_schema,name,0.95,llm_based_matching
1,dbpedia,platform,target_schema,platform,0.95,llm_based_matching
2,dbpedia,developer,target_schema,developer,0.95,llm_based_matching
3,dbpedia,releaseDate,target_schema,releaseYear,0.95,llm_based_matching
4,dbpedia,series,target_schema,series,0.95,llm_based_matching
5,dbpedia,id,target_schema,id,0.95,llm_based_matching


In [10]:
metacritic_mapping = matcher.match(metacritic, df_target)
metacritic_mapping

Invalid column mapping: genres -> genres


,source_dataset,source_column,target_dataset,target_column,score,notes
0,metacritic,name,target_schema,name,0.95,llm_based_matching
1,metacritic,release_date,target_schema,releaseYear,0.95,llm_based_matching
2,metacritic,developer,target_schema,developer,0.95,llm_based_matching
3,metacritic,platform,target_schema,platform,0.95,llm_based_matching
4,metacritic,rating,target_schema,ESRB,0.95,llm_based_matching
5,metacritic,metascore,target_schema,criticScore,0.95,llm_based_matching
6,metacritic,user_score,target_schema,userScore,0.95,llm_based_matching
7,metacritic,id,target_schema,id,0.95,llm_based_matching


In [11]:
sales_mapping = matcher.match(sales, df_target)
sales_mapping

Invalid column mapping: Genre -> genres


,source_dataset,source_column,target_dataset,target_column,score,notes
0,sales,Title,target_schema,name,0.95,llm_based_matching
1,sales,Platform,target_schema,platform,0.95,llm_based_matching
2,sales,Year_of_Release,target_schema,releaseYear,0.95,llm_based_matching
3,sales,Publisher,target_schema,publisher,0.95,llm_based_matching
4,sales,Global_Sales,target_schema,globalSales,0.95,llm_based_matching
5,sales,Critic_Score,target_schema,criticScore,0.95,llm_based_matching
6,sales,User_Score,target_schema,userScore,0.95,llm_based_matching
7,sales,Developer,target_schema,developer,0.95,llm_based_matching
8,sales,Rating,target_schema,ESRB,0.95,llm_based_matching
9,sales,id,target_schema,id,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [12]:
# Unify platform names across datasets
platform_groups = {
    "Nintendo Entertainment System": ["NES"],
    "Super Nintendo": ["SNES", "Super Nintendo Entertainment System"],
    "Nintendo 64": ["N64"],
    "GameCube": ["Nintendo GameCube", "GC"],
    "Wii": ["Nintendo Wii"],
    "Game Boy Color": ["GBC"],
    "Game Boy Advance": ["GBA"],
    "Game Boy": ["GB"],
    "DS": ["Nintendo DS"],
    "3DS": ["Nintendo 3DS"],
    "Switch": ["Nintendo Switch"],

    "Playstation": [
        "Playstation (console)", "PlayStation (console)",
        "Playstation 1", "PlayStation 1",
        "PS1", "PSX", "PS"
    ],

    "PS2": ["Playstation 2", "PlayStation 2"],
    "PS3": ["Playstation 3", "PlayStation 3"],
    "PS4": ["Playstation 4", "PlayStation 4"],
    "Playstation Portable": ["PSP"],
    "Playstation Vita": ["PSV", "PS Vita"],
    "Playstation VR": ["PSVR", "PS VR"],

    "Xbox": ["XB", "Xbox (console)"],
    "Xbox One": ["XOne"],
    "Xbox 360": ["X360"],

    "PC": ["Microsoft Windows", "Windows"],
}

platform_map = {}
for canonical, variants in platform_groups.items():
    for alias in variants:
        platform_map[alias.lower()] = canonical

def normalize_platform(series):
    def _norm(x):
        # Leave missing or non-string values as they are
        if not isinstance(x, str):
            return x
        key = x.strip().lower()
        return platform_map.get(key, x.strip())
    
    return series.apply(_norm)

dbpedia["platform"] = normalize_platform(dbpedia["platform"])
metacritic["platform"] = normalize_platform(metacritic["platform"])
sales["Platform"] = normalize_platform(sales["Platform"])

In [13]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("releaseYear", output_type="datetime")
dbpedia_normalized = translator.translate(
    dbpedia, dbpedia_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse special date format (e.g., Oct 26, 2018)
spec.set_column("releaseYear", output_type="datetime", date_format="%b %d, %Y")

metacritic_normalized = translator.translate(
    metacritic, metacritic_mapping,
    normalize=spec, on_failure="keep"
)

# Configure date column to parse year-only values (e.g., "1929", "2010")
spec.set_column("releaseYear", output_type="datetime", date_format="%Y")

sales_normalized = translator.translate(
    sales, sales_mapping,
    normalize=spec, on_failure="keep"
)

In [14]:
# Inspect normalized dbpedia dataset (target columns only)
dbpedia_cols = [c for c in target_columns if c in dbpedia_normalized.columns]
dbpedia_normalized[dbpedia_cols].head(10)

,id,name,releaseYear,developer,platform,series
0,1,San Francisco Rush 2049,2006-02-17,Handheld Games,Game Boy Color,Rush (video game series)
1,2,RoboCop (1988 video game),1989-12-12,Ocean Software,Arcade video game,List of RoboCop video games
2,3,Air (video game),2016-09-08,Key (company),PlayStation Vita,None
3,4,Fallout 2,1998-10-29,Black Isle Studios,Mac OS X,Fallout (series)
4,5,SpongeBob SquarePants: Creature from the Krust...,2006-10-18,Blitz Games,Wii,SpongeBob SquarePants video games
5,6,Transformers: Fall of Cybertron,2016-08-08,High Moon Studios,PS4,Transformers
6,7,List of Monster Jam video games,2003-12-10,Ubi Soft Barcelona,Xbox One,Monster Jam
7,8,Onimusha: Warlords,2002-01-28,Capcom,Xbox One,Onimusha
8,9,Nicktoons: Battle for Volcano Island,2006-10-24,Halfbrick,PS2,SpongeBob SquarePants video games
9,10,Grid Autosport,2019-09-19,Feral Interactive,Xbox 360,Grid (series)


In [15]:
metacritic_cols = [c for c in target_columns if c in metacritic_normalized.columns]
metacritic_normalized[metacritic_cols].head(10)

,id,name,releaseYear,developer,platform,criticScore,userScore,ESRB
0,1,Red Dead Redemption 2,2018-10-26,Rockstar Games,Xbox One,97.0,8.3,M
1,2,Grand Theft Auto IV,2008-04-29,Rockstar North,Xbox 360,98.0,8.0,M
2,3,SoulCalibur,1999-09-08,Namco,Dreamcast,98.0,8.4,T
3,4,Tony Hawk's Pro Skater 2,2000-09-20,Neversoft Entertainment,PlayStation,98.0,7.5,T
4,5,Super Mario Galaxy,2007-11-12,Nintendo,Wii,97.0,9.1,E
5,6,Grand Theft Auto IV,2008-04-29,Rockstar North,PS3,98.0,7.9,M
6,7,Call of Duty 4: Modern Warfare,2007-11-05,Infinity Ward,Xbox 360,94.0,8.5,M
7,8,The Elder Scrolls IV: Oblivion,2006-03-20,"Bethesda Softworks,Bethesda Game Studios",PC,94.0,8.3,M
8,9,Super Mario Galaxy 2,2010-05-23,Nintendo EAD Tokyo,Wii,97.0,9.1,E
9,10,The Legend of Zelda: Ocarina of Time,1998-11-23,Nintendo,Nintendo 64,99.0,9.0,E


In [16]:
sales_cols = [c for c in target_columns if c in sales_normalized.columns]
sales_normalized[sales_cols].head(10)

,id,name,releaseYear,developer,publisher,platform,criticScore,userScore,ESRB,globalSales
0,1,Wii Sports,2006-01-01,Nintendo,Nintendo,Wii,76.0,8.0,E,82
1,2,Mario Kart Wii,2008-01-01,Nintendo,Nintendo,Wii,82.0,8.3,E,35
2,3,Wii Sports Resort,2009-01-01,Nintendo,Nintendo,Wii,80.0,8.0,E,32
3,4,New Super Mario Bros.,2006-01-01,Nintendo,Nintendo,DS,89.0,8.5,E,29
4,5,Wii Play,2006-01-01,Nintendo,Nintendo,Wii,58.0,6.6,E,28
5,6,New Super Mario Bros. Wii,2009-01-01,Nintendo,Nintendo,Wii,87.0,8.4,E,28
6,7,Mario Kart DS,2005-01-01,Nintendo,Nintendo,DS,91.0,8.6,E,23
7,8,Wii Fit,2007-01-01,Nintendo,Nintendo,Wii,80.0,7.7,E,22
8,9,Kinect Adventures!,2010-01-01,Good Science Studio,Microsoft Game Studios,Xbox 360,61.0,6.3,E,21
9,10,Wii Fit Plus,2009-01-01,Nintendo,Nintendo,Wii,80.0,7.4,E,21


In [17]:
# Only keep target columns
dbpedia = dbpedia_normalized[dbpedia_cols].copy()
metacritic = metacritic_normalized[metacritic_cols].copy()
sales = sales_normalized[sales_cols].copy()

# Create proper id entries for datasets
dbpedia["id"] = dbpedia["id"].apply(lambda x: f"dbpedia_{x}")
metacritic["id"] = metacritic["id"].apply(lambda x: f"metacritic_{x}")
sales["id"] = sales["id"].apply(lambda x: f"sales_{x}")

## Part 1: Data Profiling

In [18]:
# Display basic information
datasets = [dbpedia, metacritic, sales]
names = ["DBpedia", "Metacritic", "Sales"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 82,705


In [19]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

dbpedia:
  Rows: 54,333
  Columns: 6
  Total nulls: 29,077
  Null percentage: 8.9%
  Null counts per column:
    releaseYear: 1,409 (2.6%)
    developer: 1,406 (2.6%)
    platform: 460 (0.8%)
    series: 25,802 (47.5%)

metacritic:
  Rows: 20,494
  Columns: 8
  Total nulls: 3,720
  Null percentage: 2.3%
  Null counts per column:
    releaseYear: 2 (0.0%)
    developer: 19 (0.1%)
    criticScore: 10 (0.0%)
    userScore: 1,413 (6.9%)
    ESRB: 2,276 (11.1%)

sales:
  Rows: 7,878
  Columns: 10
  Total nulls: 1
  Null percentage: 0.0%
  Null counts per column:
    publisher: 1 (0.0%)



### Attribute Coverage Analysis

In [20]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,metacritic_count,metacritic_pct,metacritic_coverage,metacritic_samples,sales_count,sales_pct,sales_coverage,sales_samples,avg_coverage,max_coverage,datasets_with_attribute
0,ESRB,0/0,0%,0.000000,N/A,18218/20494,88.9%,0.888943,"['M', 'M', 'T']",7878/7878,100.0%,1.000000,"['E', 'E', 'E']",0.629648,1.000000,2
1,criticScore,0/0,0%,0.000000,N/A,20484/20494,100.0%,0.999512,"[97.0, 98.0, 98.0]",7878/7878,100.0%,1.000000,"[76.0, 82.0, 80.0]",0.666504,1.000000,2
2,developer,52927/54333,97.4%,0.974123,"['Handheld Games', 'Ocean Software', 'Key (com...",20475/20494,99.9%,0.999073,"['Rockstar Games', 'Rockstar North', 'Namco']",7878/7878,100.0%,1.000000,"['Nintendo', 'Nintendo', 'Nintendo']",0.991065,1.000000,3
3,globalSales,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7878/7878,100.0%,1.000000,"[82, 35, 32]",0.333333,1.000000,1
4,id,54333/54333,100.0%,1.000000,"['dbpedia_1', 'dbpedia_2', 'dbpedia_3']",20494/20494,100.0%,1.000000,"['metacritic_1', 'metacritic_2', 'metacritic_3']",7878/7878,100.0%,1.000000,"['sales_1', 'sales_2', 'sales_3']",1.000000,1.000000,3
5,name,54333/54333,100.0%,1.000000,"['San Francisco Rush 2049', 'RoboCop (1988 vid...",20494/20494,100.0%,1.000000,"['Red Dead Redemption 2', 'Grand Theft Auto IV...",7878/7878,100.0%,1.000000,"['Wii Sports', 'Mario Kart Wii', 'Wii Sports R...",1.000000,1.000000,3
6,platform,53873/54333,99.2%,0.991534,"['Game Boy Color', 'Arcade video game', 'PlayS...",20494/20494,100.0%,1.000000,"['Xbox One', 'Xbox 360', 'Dreamcast']",7878/7878,100.0%,1.000000,"['Wii', 'Wii', 'Wii']",0.997178,1.000000,3
7,publisher,0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,7877/7878,100.0%,0.999873,"['Nintendo', 'Nintendo', 'Nintendo']",0.333291,0.999873,1
8,releaseYear,52924/54333,97.4%,0.974067,"[Timestamp('2006-02-17 00:00:00'), Timestamp('...",20492/20494,100.0%,0.999902,"[Timestamp('2018-10-26 00:00:00'), Timestamp('...",7878/7878,100.0%,1.000000,"[Timestamp('2006-01-01 00:00:00'), Timestamp('...",0.991323,1.000000,3
9,series,28531/54333,52.5%,0.525114,"['Rush (video game series)', 'List of RoboCop ...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.175038,0.525114,1



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']


## Part 2: Entity Matching

### Step 1: Blocking

In [21]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [22]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

dbpedia['name_longest_token'] = dbpedia['name'].apply(get_longest_token)
metacritic['name_longest_token'] = metacritic['name'].apply(get_longest_token)
sales['name_longest_token'] = sales['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    metacritic, dbpedia,
    on=['name_longest_token', 'platform'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_m2d = standard_blocker_m2d.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 13482 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 18797 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 5915 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv


### Step 2: Evaluate Blocking Against Ground Truth

In [23]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/aaronsteiner/Documents/GitHub/unsupervised-data-integration/usecases/input/games/entitymatching/metacritic_2_dbpedia_test.csv'

In [ ]:
standard_blocker_m2s = StandardBlocker(
    metacritic, sales,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_sales_test.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2s,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 5497 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2231 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2100 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 2 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 5 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 8 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 10 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 12 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 16 true matches
[INFO ] root - Proc

{'pair_completeness': 0.8431372549019608,
 'pair_quality': 0.0007636073045846035,
 'reduction_ratio': 0.9989536501225023,
 'total_candidates': 168935,
 'total_possible_pairs': 161451732,
 'true_positives_found': 129,
 'total_true_pairs': 153,
 'batches_processed': 169,
 'evaluation_timestamp': '2025-12-10T14:33:06.351411',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/blocking-evaluation/blocking_detailed_results.csv']}

Now let's evaluate which blocking method we want to use for each dataset combination:

### Step 3: Entity Matching with Comparators

In [ ]:
from PyDI.entitymatching import StringComparator, DateComparator

# Create comparators for different attributes
comparators_m2d = [
    # Name similarity - most important for games
    StringComparator(
        column='name',
        similarity_function='jaccard',  # Good for game names
        preprocess=str.lower  # Case normalization
    ),
    
    # Platform similarity - supporting evidence
    StringComparator(
        column='developer',
        similarity_function='jaccard',
        preprocess=str.lower
    ),

    # Date proximity - games from same year likely same game
    DateComparator(
        column='releaseYear'
    )
]

comparators_m2s = [
    StringComparator(
        column='name',
        similarity_function='jaccard',
        preprocess=str.lower
    ),
        # Platform similarity - supporting evidence
    StringComparator(
        column='platform',
        similarity_function='jaccard',
    ),
    DateComparator(
        column='releaseYear',
        max_days_difference=360  # Allow almost 1 year difference
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=metacritic,
    df_right=dbpedia, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators_m2d,
    weights=[0.6, 0.3, 0.1], # name, developer, releaseYear
    threshold=0.9, # set a similarity threshold for a match
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 54333 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 54333 elements after 0:00:0.091; 69706 blocked pairs (reduction ratio: 0.9999373992199602)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:25.506; found 4006 correspondences.


In [ ]:
correspondences_m2s = matcher.match(
    df_left=metacritic,
    df_right=sales, 
    candidates=standard_blocker_m2s,
    comparators=comparators_m2s,
    weights=[0.6, 0.3, 0.1], # name, platform, releaseYear
    threshold=0.8,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 20494 x 7878 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 20494 x 7878 elements after 0:00:0.120; 168935 blocked pairs (reduction ratio: 0.9989536501225023)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:63.636; found 6546 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_dbpedia_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  179
[INFO ] root -   True Negatives:  393
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 27
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.955
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.869
[INFO ] root -   F1-Score:  0.930


{'precision': 1.0,
 'recall': 0.8689320388349514,
 'f1': 0.9298701298701298,
 'accuracy': 0.9549248747913188,
 'true_positives': 179,
 'false_positives': 0,
 'false_negatives': 27,
 'true_negatives': 393,
 'threshold_used': 0.0,
 'total_correspondences': 4006,
 'filtered_correspondences': 4006,
 'evaluation_timestamp': '2025-12-10T14:34:45.373845',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/games/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/games/debug_results_entity_matching/matching_detailed_results.csv']}

In [ ]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2735 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	1965	|	71.85%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	470	|	17.18%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	175	|	6.40%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	78	|	2.85%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	31	|	1.13%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	9	|	0.33%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	4	|	0.15%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	1	|	0.04%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	1	|	0.04%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.04%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/cluster_size_distribution.csv


In [ ]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/games/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 2735 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [ ]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm
     
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_m2d = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 4006 -> 4006 (threshold=0.0)
[INFO ] root - Greedy matching: 4006 -> 2735 correspondences (5470 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 4006 -> 2735 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 6741 -> 5470 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2735 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2735	|	100.00%
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  149
[INFO ] root -   True Negatives:  393
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 57
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.905
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.723
[INFO ] root -   F1-Score:  0.839


In [ ]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "metacritic_2_sales_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2s = clusterer.cluster(correspondences_m2s)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2s,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2s,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  122
[INFO ] root -   True Negatives:  422
[INFO ] root -   False Positives: 25
[INFO ] root -   False Negatives: 31
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.907
[INFO ] root -   Precision: 0.830
[INFO ] root -   Recall:    0.797
[INFO ] root -   F1-Score:  0.813
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 6305 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	6197	|	98.29%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	27	|	0.43%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	74	|	1.17%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	2	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	5	|	0.08%
[INFO ] root - Filtered correspondences: 6546 -> 6546 (threshold=0.0)
[INFO ] root - Maximum bip

## Part 3: Data Fusion

In [ ]:
metacritic["metacritic_id"] = metacritic["id"]

# Assign trust scores to datasets
metacritic.attrs["trust_score"] = 3
sales.attrs["trust_score"] = 2
dbpedia.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2s], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 9,126


## Step 1: Define Fusion Strategy 

In [ ]:
from PyDI.fusion import DataFusionStrategy, longest_string, prefer_higher_trust, voting, average

strategy = DataFusionStrategy('game_fusion_strategy')
['ESRB', 'criticScore', 'developer', 'id', 'name', 'platform', 'releaseYear', 'userScore']
strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('platform', voting)
strategy.add_attribute_fuser('developer', longest_string)
strategy.add_attribute_fuser('releaseYear', voting, trust_key="trust_score")
strategy.add_attribute_fuser('ESRB', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('criticScore', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('userScore', average)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'platform' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'developer' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'releaseYear' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'ESRB' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'criticScore' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'userScore' using rule 'average'


## Step 2: Run Fusion

In [ ]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[metacritic, dbpedia, sales],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'game_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 16887 of 16887 unique IDs
[INFO ] PyDI.fusion.engine - Created 73579 record groups from 9126 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 73579 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	6396	|	8.69%
[INFO ] PyDI.fusion.engine - 		3	|	1365	|	1.86%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     ESRB: 1.00
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     criticScor

Fused rows: 7,761


,_id,_fusion_sources,_fusion_source_datasets,publisher,platform,globalSales,developer,ESRB,metacritic_id,userScore,releaseYear,name_longest_token,id,name,criticScore,_fusion_confidence,_fusion_metadata,series
0,metacritic_1512,"[metacritic_1512, sales_85]","[metacritic, sales]",Electronic Arts,PS3,6.0,EA Canada,E,metacritic_1512,4.35,2013-09-24,FIFA,metacritic_1512,FIFA 14,86.0,0.726228,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
1,metacritic_5371,"[metacritic_5371, dbpedia_51865, sales_4297]","[metacritic, dbpedia, sales]",Konami Digital Entertainment,Wii,0.0,Konami,E,metacritic_5371,7.25,2011-11-15,Evolution,metacritic_5371,Pro Evolution Soccer 2012,79.0,0.705460,"{'publisher_rule': 'first_non_null', 'publishe...",Pro Evolution Soccer
2,metacritic_2165,"[metacritic_2165, sales_939]","[metacritic, sales]",Capcom,Xbox 360,1.0,Capcom,M,metacritic_2165,8.20,2008-02-05,Devil,metacritic_2165,Devil May Cry 4,84.0,0.725055,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
3,sales_4738,"[sales_4738, metacritic_16469]","[sales, metacritic]",Vivendi Games,GameCube,0.0,Inevitable Entertainment,E,metacritic_16469,7.75,2003-01-01,Hobbit,sales_4738,The Hobbit,61.0,0.721994,"{'publisher_rule': 'first_non_null', 'publishe...",NaN
4,dbpedia_15997,"[dbpedia_15997, sales_918, metacritic_4786]","[dbpedia, sales, metacritic]",Capcom,PS3,1.0,Blue Castle Games,M,metacritic_4786,6.75,2010-09-24,Rising,dbpedia_15997,Dead Rising 2,80.0,0.707716,"{'publisher_rule': 'first_non_null', 'publishe...",Dead Rising


## Step 3: Evaluate Data Fusion

In [ ]:
from PyDI.fusion import tokenized_match, year_only_match, boolean_match, numeric_tolerance_match, exact_match
strategy.add_evaluation_function("name", exact_match)
strategy.add_evaluation_function("platform", exact_match)
strategy.add_evaluation_function("developer", exact_match)
strategy.add_evaluation_function("releaseYear", year_only_match)
strategy.add_evaluation_function("ESRB", exact_match)
strategy.add_evaluation_function("criticScore", numeric_tolerance_match, tolerance=2)
strategy.add_evaluation_function("userScore", numeric_tolerance_match, tolerance=0.2)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'platform'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'developer'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'releaseYear'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'ESRB'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'criticScore' with params {'tolerance': 2}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'userScore' with params {'tolerance': 0.2}


In [ ]:
# Evaluate fusion results against validation set in order to make adjustments if necessary
from PyDI.fusion import DataFusionEvaluator
from PyDI.io import load_xml

fusion_val_set = load_xml(INPUT_DIR / 'fusion' / 'validation_set.xml', name='fusion_val_set', nested_handling='aggregate')
# transform releaseYear column to datetime
fusion_val_set['releaseYear'] = pd.to_datetime(fusion_val_set['releaseYear'],errors='coerce')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the validation set
print("Evaluating fusion results against validation set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_val_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Validation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[WARNING] PyDI.fusion.evaluation - Missing 1 expected/reference records in fused dataset: metacritic_1593
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.825 overall accuracy (66/80)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 14 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |       4 |     28.57%%
[INFO ] PyDI.fusion.evaluation - 	publisher                        |       2 |     14.29%%
[INFO ] PyDI.fusion.evaluation - 	ESRB                             |       2 |     14.29%%
[INFO ] PyDI.f

Evaluating fusion results against validation set...

Fusion Validation Results:
  overall_accuracy: 0.825
  macro_accuracy: 0.825
  num_evaluated_records: 10
  num_evaluated_attributes: 8
  total_evaluations: 80
  total_correct: 66
  publisher_accuracy: 0.800
  publisher_count: 10
  platform_accuracy: 0.900
  platform_count: 10
  name_accuracy: 0.900
  name_count: 10
  ESRB_accuracy: 0.800
  ESRB_count: 10
  userScore_accuracy: 0.600
  userScore_count: 10
  releaseYear_accuracy: 0.800
  releaseYear_count: 10
  developer_accuracy: 0.900
  developer_count: 10
  criticScore_accuracy: 0.900
  criticScore_count: 10

Overall Accuracy: 82.5%


In [ ]:
# Finally, evaluate against test set
fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
fusion_test_set['releaseYear'] = pd.to_datetime(fusion_test_set['releaseYear'],errors='coerce')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the test set
print("Evaluating fusion results against test set...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='metacritic_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Test Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/games/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.840 overall accuracy (100/119)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 19 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	userScore                        |      10 |     52.63%%
[INFO ] PyDI.fusion.evaluation - 	developer                        |       3 |     15.79%%
[INFO ] PyDI.fusion.evaluation - 	publisher                        |       2 |     10.53%%
[INFO ] PyDI.fusion.evaluation - 	name                             |       2 |     10.53%%
[INFO ] PyDI.fusion.evaluat

Evaluating fusion results against test set...

Fusion Test Results:
  overall_accuracy: 0.840
  macro_accuracy: 0.836
  num_evaluated_records: 15
  num_evaluated_attributes: 8
  total_evaluations: 119
  total_correct: 100
  publisher_accuracy: 0.867
  publisher_count: 15
  platform_accuracy: 1.000
  platform_count: 15
  name_accuracy: 0.867
  name_count: 15
  ESRB_accuracy: 0.867
  ESRB_count: 15
  userScore_accuracy: 0.286
  userScore_count: 14
  releaseYear_accuracy: 1.000
  releaseYear_count: 15
  developer_accuracy: 0.800
  developer_count: 15
  criticScore_accuracy: 1.000
  criticScore_count: 15

Overall Accuracy: 84.0%
